# Classic FiLM UNet: modulation analysis

Same experiment menu as the Attn notebook (descriptive, probing, interventions, OOD, interpolation, analogy, GANSpace PCA, TCAV-on-FiLM) but adapted for `UNet2DFiLM` whose γ/β depend **only on organ_id** — image is irrelevant to the modulation.

Key consequences (these are sanity checks more than discoveries):
- Within-organ variance of γ/β is **exactly zero**.
- 6.1 organ-mean swap should produce **ΔDice = 0** by construction.
- 5.2 Dice ← γ regression is degenerate (constant γ per organ).
- TCAV / probes still work because γ varies across organs.

Each FiLM2d block has its **own** organ embedding (no globally-shared one). We use the **first FiLM2d block's** embedding weights as a representative for sections that need an organ→vector mapping (4.4 PCA, 8 interpolation, 9 analogy). For interventions, the embedding sample is shared across all blocks (each block's MLP reinterprets it).

## 0. Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, os
from pathlib import Path
REPO = Path('..').resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import torch
import matplotlib.pyplot as plt
from safetensors.torch import load_file
from torch.utils.data import DataLoader

import notebooks._film_analysis as fa
from nets.segm_net import UNet2DFiLM
from data_classes.datasets import USdatasetOmni
from utils.utils import get_sft_transforms, organ_to_class_dict, class_to_organ_dict
from utils.paths import DATA_DIR

torch.manual_seed(0); np.random.seed(0)
FIG_DIR = Path('figures/film'); FIG_DIR.mkdir(parents=True, exist_ok=True)
print('Repo:', REPO)
print('Data:', DATA_DIR, '(exists:', Path(DATA_DIR).exists(), ')')
print('Figures →', FIG_DIR.resolve())

## 1. Model + checkpoint

In [ ]:
CHECKPOINT = './loggings/1502cf4f7049/checkpoint-9200/model.safetensors'
# CHECKPOINT = './loggings/0c0d2b9760e3/checkpoint-9200/model.safetensors'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

model = UNet2DFiLM(
    in_channels=3, num_classes=1, n_organs=10,
    size=16, depth=5,
    film_start=0, use_film=True,
    distill=False,
)

state = load_file(CHECKPOINT)
state = {k: v for k, v in state.items() if 'distill' not in k}
res = model.load_state_dict(state, strict=False)
print('missing:', len(res.missing_keys), 'unexpected:', len(res.unexpected_keys))
model.eval().to(DEVICE)
print('max_channels:', model.size * (2 ** (model.depth + 1)))
print('layer_configs:', model._build_layer_configs())
print('film_embed:', model.film_embed)

## 2. Val data

In [ ]:
IMG_SIZE = 512
BATCH = 8
assert Path(DATA_DIR).exists(), f'DATA_DIR {DATA_DIR} not found; update utils/paths.py'

test_ds = USdatasetOmni(
    DATA_DIR, split='val_cls',
    transforms=get_sft_transforms(train=False, size=IMG_SIZE),
    data_type='segmentation',
    out_size=IMG_SIZE,
    keep_aspect_ratio=True, self_norm=True
)
loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=4,
                    pin_memory=DEVICE == 'cuda')
print('test set:', len(test_ds), 'samples')

In [ ]:
nuts_cnt = 0
for item in test_ds.items[:]:
    if item['organ_label'] == 'testicle':
        nuts_cnt+=1
        if nuts_cnt >= 100: test_ds.items.remove(item)


## 3. Modulation collection

Same collector. Because the FiLM model's modulation is image-invariant, γ/β within a fixed organ_id will be **identical** across samples. We assert this below as a correctness check.

In [ ]:
MAX_BATCHES = 200
# run_forward=False skips the UNet forward pass so this cell completes in seconds.
# Dice scores are computed separately before section 5.2.
data = fa.collect_modulations(model, loader, DEVICE, max_batches=MAX_BATCHES, run_forward=False)
print('Samples:', len(data['organ_id']), '| layers:', len(data['layers']))

# Sanity check: within-organ γ variance should be ~0 (FiLM model is image-invariant)
for lid in sorted(data['layers'].keys())[:3]:
    G = data['layers'][lid]['gamma']
    organs = [int(o) for o in np.unique(data['organ_id']) if o >= 0]
    max_within = 0.0
    for o in organs:
        m = data['organ_id'] == o
        if m.sum() < 2: continue
        max_within = max(max_within, float(G[m].var(axis=0).max()))
    print(f'L{lid:2d} max within-organ γ variance: {max_within:.3e}  (should be ~0)')

## 4. Tier 1 — descriptive

In [ ]:
# 4.1 γ/β histograms per layer
layer_ids = sorted(data['layers'].keys())
n = len(layer_ids)
fig, axes = plt.subplots(2, n, figsize=(2.2*n, 4.4), sharey='row')
for i, lid in enumerate(layer_ids):
    g = data['layers'][lid]['gamma'].ravel()
    b = data['layers'][lid]['beta'].ravel()
    axes[0, i].hist(g, bins=80, color='steelblue'); axes[0, i].set_title(f'L{lid} γ', fontsize=9)
    axes[0, i].axvline(0, color='k', lw=.5); axes[0, i].axvline(1, color='r', lw=.5)
    axes[1, i].hist(b, bins=80, color='salmon'); axes[1, i].set_title(f'L{lid} β', fontsize=9)
    axes[1, i].axvline(0, color='k', lw=.5)
    for a in (axes[0,i], axes[1,i]): a.tick_params(labelsize=7)
fig.suptitle('γ / β histograms per layer (FiLM)')
fig.tight_layout(); fig.savefig(FIG_DIR/'4_1_hists.png', dpi=130); plt.show()

In [ ]:
# 4.2 Per-channel selectivity per layer.
# Because within-organ variance is ~0, selectivity = Var_across_organ / ε is effectively unbounded
# wherever γ differs across organs. We instead report Var_across_organ directly.
var_per_layer = {}
for lid in layer_ids:
    G = data['layers'][lid]['gamma']
    organs = [int(o) for o in np.unique(data['organ_id']) if o >= 0]
    organ_means = np.stack([G[data['organ_id']==o].mean(axis=0) for o in organs])  # (n_organs, C)
    var_per_layer[lid] = organ_means.var(axis=0)
fig, ax = plt.subplots(figsize=(8, 4))
for lid, v in var_per_layer.items():
    ax.plot(np.sort(v)[::-1], label=f'L{lid}', lw=1)
ax.set_yscale('log'); ax.set_xlabel('channel rank'); ax.set_ylabel('Var across organs (log)')
ax.legend(fontsize=7, ncol=3, loc='upper right')
ax.set_title('Per-channel across-organ γ variance (proxy for selectivity)')
fig.tight_layout(); fig.savefig(FIG_DIR/'4_2_selectivity.png', dpi=130); plt.show()

print('Channels with Var > 1e-3 (clearly organ-differentiated):')
for lid in layer_ids:
    frac = float((var_per_layer[lid] > 1e-3).mean())
    print(f'  L{lid:2d}: {frac:.2%}')

In [ ]:
# 4.3 γ-β scatter per layer: one point per organ (no within-organ spread)
organs_seen = sorted([int(o) for o in np.unique(data['organ_id']) if o >= 0])
cmap = plt.cm.tab10
TOP = 3
fig, axes = plt.subplots(len(layer_ids), TOP, figsize=(2.4*TOP, 2.0*len(layer_ids)))
axes = np.atleast_2d(axes)
for r, lid in enumerate(layer_ids):
    top_ch = np.argsort(var_per_layer[lid])[::-1][:TOP]
    G = data['layers'][lid]['gamma']
    B = data['layers'][lid]['beta']
    for c, ch in enumerate(top_ch):
        ax = axes[r, c]
        for oi, o in enumerate(organs_seen):
            m = data['organ_id'] == o
            ax.scatter(G[m, ch].mean(), B[m, ch].mean(), s=60, color=cmap(oi),
                       label=class_to_organ_dict.get(o, str(o)) if r==0 and c==0 else None,
                       edgecolor='k')
        ax.set_title(f'L{lid} ch{ch}', fontsize=8)
        ax.tick_params(labelsize=6)
        if c == 0: ax.set_ylabel('β', fontsize=7)
        if r == len(layer_ids)-1: ax.set_xlabel('γ', fontsize=7)
fig.suptitle('γ-β per organ — top-3 most variant channels per layer')
if axes[0,0].get_legend_handles_labels()[0]:
    fig.legend(*axes[0,0].get_legend_handles_labels(), loc='upper right', fontsize=7)
fig.tight_layout(); fig.savefig(FIG_DIR/'4_3_scatter.png', dpi=130); plt.show()

In [ ]:
# 4.4 Organ embedding PCA — first FiLM2d block as representative
from sklearn.decomposition import PCA
film_layers = model._get_film_layers_in_order()
rep_block = film_layers[0]
emb = rep_block.embed.weight.detach().cpu().numpy()[organs_seen]  # (n_organs+1, film_embed)

pca = PCA(n_components=2).fit(emb)
proj = pca.transform(emb)
fig, ax = plt.subplots(figsize=(6, 5))
for i in range(emb.shape[0]):
    label = class_to_organ_dict.get(organs_seen[i], 'unknown' if i == emb.shape[0]-1 else f'id{i}')
    ax.scatter(proj[i,0], proj[i,1], s=80,
               color='red' if i == emb.shape[0]-1 else cmap(i % 10),
               marker='X' if i == emb.shape[0]-1 else 'o',
               edgecolor='k')
    ax.annotate(label, (proj[i,0], proj[i,1]), fontsize=8, xytext=(4,4), textcoords='offset points')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.0%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.0%})')
ax.set_title('First FiLM block organ embedding — PCA')
fig.tight_layout(); fig.savefig(FIG_DIR/'4_4_organ_embed_pca.png', dpi=130); plt.show()

In [ ]:
# 4.5 γ-vector PCA per organ (GANSpace-style: PCA in modulation space)
proj, pca, organs_used = fa.pca_on_gammas(data['layers'], data['organ_id'], n_components=4)
fig, ax = plt.subplots(figsize=(6, 5))
for i, o in enumerate(organs_used):
    ax.scatter(proj[i, 0], proj[i, 1], s=120, color=cmap(i % 10), edgecolor='k')
    ax.annotate(class_to_organ_dict.get(o, str(o)), (proj[i,0], proj[i,1]),
                fontsize=9, xytext=(5,5), textcoords='offset points')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.0%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.0%})')
ax.set_title('PCA of per-organ mean γ vectors (concatenated across FiLM layers)')
fig.tight_layout(); fig.savefig(FIG_DIR/'4_5_gamma_pca.png', dpi=130); plt.show()
print('Cumulative explained var:', np.cumsum(pca.explained_variance_ratio_).round(3))

## 5. Tier 2 — linear probing

In [ ]:
# Compute dice scores — only needed for sections 5.2 and the summary.
# Runs the full UNet forward pass over the dataset (~1–3 min depending on GPU).
data['dice'] = fa.compute_dice_scores(model, loader, DEVICE, max_batches=MAX_BATCHES)

means = []
for i in np.unique(data['organ_id']):
    if i == 7: continue
    means.append((data['dice'][data['organ_id'] == i]).mean())
    print(f"Organ: {i}, # cases: {len((data['dice'][data['organ_id'] == i]))}, mean dice:{means[-1]}")
print(f"Mean Dice (mean of organ means): {np.mean(means):.4f}")

In [ ]:
# 5.1 Linear probe organ_id ← γ at each layer.
# Because γ is constant per organ, probe accuracy will be essentially perfect at every layer where γ differs across organs.
probe_acc = {}
for lid in layer_ids:
    X = data['layers'][lid]['gamma']
    res = fa.fit_linear_probe(X, data['organ_id'])
    probe_acc[lid] = res
    print(f'L{lid:2d} probe acc: {res["acc"]:.3f} ± {res.get("std",0):.3f}')
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(list(probe_acc.keys()), [v['acc'] for v in probe_acc.values()], 'o-')
ax.set_xlabel('layer id'); ax.set_ylabel('5-fold accuracy')
ax.axhline(1/len([o for o in np.unique(data['organ_id']) if o>=0]), color='k', ls='--', lw=.5, label='chance')
ax.set_title('Linear probe: organ_id from γ at each layer (expect ≈1.0)'); ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR/'5_1_probe.png', dpi=130); plt.show()

In [ ]:
# 5.2 Dice ← γ regression — degenerate for FiLM (γ is constant per organ).
# Reported only as a sanity check that R² is near zero variance-explained (since X has no variance
# beyond organ_id, which is itself a strong predictor of Dice via organ difficulty).
valid = data['organ_id'] >= 0
for o in sorted([int(o) for o in np.unique(data['organ_id'][valid])]):
    m = data['organ_id'] == o
    if m.sum() < 20: continue
    Xs = [data['layers'][lid]['gamma'][m] for lid in layer_ids]
    X = np.concatenate(Xs, axis=1)
    res = fa.fit_dice_regressor(X, data['dice'][m])
    print(f'organ {o} ({class_to_organ_dict.get(o,"?")}, n={m.sum()}): {res}')

## 6. Tier 3 — causal interventions

In [ ]:
g_per_layer_per_organ = []
b_per_layer_per_organ = []
for lid in sorted(data['layers'].keys()):
    g_per, b_per = fa.per_organ_mean_gamma(data['layers'][lid], data['organ_id'])
    g_per_layer_per_organ.append(g_per)
    b_per_layer_per_organ.append(b_per)
layer_configs = model._build_layer_configs()
MAX_C = model.size * (2 ** (model.depth + 1))
INTERVENE_BATCHES = 200

In [ ]:
# 6.1 Organ-mean swap — sanity check: ΔDice should be ~0 for the FiLM model
baseline_dices, swapped_dices, organ_ids_6 = [], [], []
for bi, batch in enumerate(loader):
    if bi >= INTERVENE_BATCHES: break
    pv = batch['pixel_values'].to(DEVICE); m = batch['masks'].to(DEVICE)
    oid = batch['organ_id'].to(DEVICE).long()
    with torch.no_grad():
        base = model(pixel_values=pv, organ_id=oid, masks=m)
    baseline_dices.append(fa.compute_per_sample_dice(base['logits'], m).cpu().numpy())
    organ_ids_6.append(oid.cpu().numpy())
    mod_list = fa.build_mod_list_from_per_organ(
        oid.cpu(), g_per_layer_per_organ, b_per_layer_per_organ,
        layer_configs, MAX_C, DEVICE,
    )
    _, dice_sw = fa.run_with_modulations(model, batch, mod_list, DEVICE)
    swapped_dices.append(dice_sw.cpu().numpy())
baseline = np.concatenate(baseline_dices); swapped = np.concatenate(swapped_dices)
organ_ids_6 = np.concatenate(organ_ids_6)

# def _organ_mean(dices, oids):
#     return np.mean([dices[oids == o].mean() for o in np.unique(oids) if o >= 0 and o != 7 and (oids == o).sum() > 0])

def _organ_mean(dices, oids):
    return np.mean([dices[oids == o].mean() for o in np.unique(oids) if o >= 0 and o != 7 and (oids == o).sum() > 0])

baseline_m = _organ_mean(baseline, organ_ids_6)
swapped_m  = _organ_mean(swapped, organ_ids_6)
print(f'baseline Dice: {baseline_m:.4f}  organ-mean swap Dice: {swapped_m:.4f}  Δ: {swapped_m-baseline_m:+.4f}')
print(f'Max |Δ| per sample: {np.abs(baseline-swapped).max():.4e}  (should be ~0)')
fig, ax = plt.subplots(figsize=(5,3))
ax.hist(baseline-swapped, bins=40); ax.axvline(0, color='r', lw=.8)
ax.set_xlabel('Dice(baseline) − Dice(organ-mean swap)'); ax.set_title('Δ from removing image-instance modulation')
fig.tight_layout(); fig.savefig(FIG_DIR/'6_1_organ_mean_swap.png', dpi=130); plt.show()

In [ ]:
# 6.2 Cross-organ swap: image of organ A, modulation of organ B's mean
organs_target = organs_used
K = len(organs_target)
dice_matrix = np.full((K, K), np.nan)
batches_cache = []
for bi, batch in enumerate(loader):
    if bi >= INTERVENE_BATCHES: break
    batches_cache.append(batch)
for ai, organ_a in enumerate(organs_target):
    for bi_, organ_b in enumerate(organs_target):
        dices = []
        for batch in batches_cache:
            oid = batch['organ_id'].long()
            sel = (oid == organ_a)
            if sel.sum() == 0: continue
            sub = {k: v[sel] for k, v in batch.items() if torch.is_tensor(v)}
            B = sub['pixel_values'].shape[0]
            mod_list = fa.build_mod_list_from_per_organ(
                torch.full((B,), organ_b, dtype=torch.long),
                g_per_layer_per_organ, b_per_layer_per_organ,
                layer_configs, MAX_C, DEVICE,
            )
            _, d = fa.run_with_modulations(model, sub, mod_list, DEVICE)
            dices.append(d.cpu().numpy())
        if dices:
            dice_matrix[ai, bi_] = float(np.concatenate(dices).mean())
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(dice_matrix, vmin=0, vmax=1, cmap='viridis')
labels = [class_to_organ_dict.get(o, str(o)) for o in organs_target]
ax.set_xticks(range(K)); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticks(range(K)); ax.set_yticklabels(labels)
ax.set_xlabel('modulation organ (B)'); ax.set_ylabel('image organ (A)')
ax.set_title('Dice when feeding organ-A images with organ-B modulation (FiLM)')
fig.colorbar(im, ax=ax); fig.tight_layout(); fig.savefig(FIG_DIR/'6_2_cross_organ.png', dpi=130); plt.show()

In [ ]:
# 6.3 Layer-wise ablation
layer_ablate_dice = {}
for target_lid in layer_ids:
    dices, oids_abl = [], []
    for bi, batch in enumerate(loader):
        if bi >= INTERVENE_BATCHES: break
        oid = batch['organ_id'].long()
        mod_list = fa.build_mod_list_from_per_organ(
            oid, g_per_layer_per_organ, b_per_layer_per_organ,
            layer_configs, MAX_C, DEVICE,
        )
        _, n_ch = layer_configs[target_lid]
        B = batch['pixel_values'].shape[0]
        mod_list[target_lid] = (torch.ones(B, n_ch, 1, 1, device=DEVICE),
                                torch.zeros(B, n_ch, 1, 1, device=DEVICE))
        _, d = fa.run_with_modulations(model, batch, mod_list, DEVICE)
        dices.append(d.cpu().numpy())
        oids_abl.append(oid.numpy())
    if dices:
        layer_ablate_dice[target_lid] = _organ_mean(np.concatenate(dices), np.concatenate(oids_abl))
    else:
        layer_ablate_dice[target_lid] = float('nan')
fig, ax = plt.subplots(figsize=(7,3))
ax.bar(list(layer_ablate_dice.keys()), [v for v in layer_ablate_dice.values()])
ax.axhline(baseline_m, color='r', ls='--', label=f'baseline 77.2')
ax.set_xlabel('ablated layer');
ax.set_ylabel('Dice'); ax.set_title('Dice after replacing one layer\'s modulation with identity (FiLM)'); ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR/'6_3_layer_ablate.png', dpi=130); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7,3))
ax.bar(list(layer_ablate_dice.keys()), [v for v in layer_ablate_dice.values()])
ax.axhline(baseline_m, color='r', ls='--', label=f'baseline 77.2')
# ax.set_xlabel('ablated layer');

ax.set_ylabel('Dice'); 
ax.set_title('Dice after replacing one layer\'s modulation with identity (FiLM)'); ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR/'6_3_layer_ablate.png', dpi=130); plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11,3))

# 1. Plot your bar chart and baseline on the original left axis
ax.bar(list(layer_ablate_dice.keys()), [v for v in layer_ablate_dice.values()])
ax.axhline(baseline_m, color='r', ls='--', label='baseline 77.2')

ax.set_ylabel('Dice')
ax.set_title("Dice after replacing one layer's modulation with identity (FiLM)")
ax.legend(loc='upper left') # Forced to left so it doesn't clash with right label

# 2. Create a twin axis on the right
ax_right = ax.twinx()

# 3. Set the label for the right axis
ax_right.set_ylabel('Base Conditioning')

# 4. Hide the tick marks/labels on the right axis so it stays clean 
# (Otherwise it will show a default 0.0 to 1.0 scale)
ax_right.set_yticks([])
ax.legend(loc='lower right')
# Use bbox_inches='tight' to make sure the new right label isn't cut off
fig.savefig(FIG_DIR/'6_3_layer_ablate_film.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# 6.4 γ-only vs β-only ablation
def run_partial(zero_gamma: bool, zero_beta: bool):
    out_dices, out_oids = [], []
    for bi, batch in enumerate(loader):
        if bi >= INTERVENE_BATCHES: break
        oid = batch['organ_id'].long()
        mod_list = fa.build_mod_list_from_per_organ(
            oid, g_per_layer_per_organ, b_per_layer_per_organ,
            layer_configs, MAX_C, DEVICE,
        )
        if zero_gamma:
            mod_list = [(torch.ones_like(g), b) for (g, b) in mod_list]
        if zero_beta:
            mod_list = [(g, torch.zeros_like(b)) for (g, b) in mod_list]
        _, d = fa.run_with_modulations(model, batch, mod_list, DEVICE)
        out_dices.append(d.cpu().numpy())
        out_oids.append(oid.numpy())
    return _organ_mean(np.concatenate(out_dices), np.concatenate(out_oids))

full = run_partial(False, False)
gamma_only = run_partial(False, True)
beta_only = run_partial(True, False)
neither = run_partial(True, True)
print(f'full mod : {full:.4f}\nγ only (β=0): {gamma_only:.4f}\nβ only (γ=1): {beta_only:.4f}\nno mod   : {neither:.4f}')

## 7. Tier 4 — OOD / embedding sampling

The FiLM model has **per-block** organ embeddings (each `FiLM2d.embed`). We sample a single embedding tensor `(B, film_embed)` and run it through each block's MLP — each block's MLP reinterprets the same vector. The 'unknown' built-in path uses `organ_id=-1` (last embedding row per block, native to the model).

In [ ]:
OOD_BATCHES = 200
EMB_DIM = model.film_embed
# Use first block's embedding weights for empirical posterior
rep_W = film_layers[0].embed.weight.detach().cpu().numpy()  # (n_organs+1, film_embed)
mu_emp, L_emp = fa.empirical_gaussian_params(film_layers[0].embed.weight)

@torch.inference_mode()
def eval_with_injected_embedding(emb_fn):
    all_dices, all_oids = [], []
    for bi, batch in enumerate(loader):
        if bi >= OOD_BATCHES: break
        pv = batch['pixel_values'].to(DEVICE); m = batch['masks'].to(DEVICE)
        B = pv.shape[0]
        emb = emb_fn(B, DEVICE)
        mod_list = fa.film_modulations_from_embedding(model, emb)
        with fa.inject_modulations(model, mod_list):
            out = model(pixel_values=pv, organ_id=batch['organ_id'].to(DEVICE).long(), masks=m)
        all_dices.append(fa.compute_per_sample_dice(out['logits'], m).cpu().numpy())
        all_oids.append(batch['organ_id'].numpy())
    if not all_dices:
        return float('nan')
    return _organ_mean(np.concatenate(all_dices), np.concatenate(all_oids))

In [ ]:
# 7.1–7.4 per-organ evaluation
# Iterates the *entire* loader so every organ in test_ds appears. To keep runtime
# bounded on over-represented organs, samples per organ are capped at MAX_PER_ORGAN.
import collections, pandas as pd

MAX_PER_ORGAN = None      # per-strategy cap. Set to None for "use everything".

# 7.1 init prior N(0, 0.02)
def emb_init(B, dev):
    return torch.randn(B, EMB_DIM, device=dev) * 0.02
# 7.2 empirical posterior (fitted on first FiLM block's embedding rows)
def emb_emp(B, dev):
    s = fa.sample_empirical_posterior(mu_emp, L_emp, B)
    return torch.as_tensor(s, dtype=torch.float32, device=dev)
# 7.3 unknown token — per-block last embedding row
def emb_unk(B, dev):
    return film_layers[0].embed.weight[-1:].expand(B, -1).to(dev)
# 7.4 nearest organ
def emb_nearest(B, dev):
    s = fa.sample_empirical_posterior(mu_emp, L_emp, B)
    nearest = fa.nearest_organ_by_cosine(s, rep_W)
    return torch.as_tensor(rep_W[nearest], dtype=torch.float32, device=dev)

strategies = {
    'baseline (true id)':  None,
    '7.1 init N(0,0.02)':  emb_init,
    '7.2 empirical post':  emb_emp,
    '7.3 unknown token':   emb_unk,
    '7.4 nearest organ':   emb_nearest,
}

@torch.inference_mode()
def eval_per_organ(strategies, max_per_organ=None):
    by_strat = {name: collections.defaultdict(list) for name in strategies}
    primary = next(iter(strategies))  # use one strategy's counts as the gate
    for bi, batch in enumerate(loader):
        oid_np = batch['organ_id'].numpy().astype(int)
        # Filter the batch to samples whose organ hasn't hit the cap yet
        if max_per_organ is None:
            keep = oid_np >= 0
        else:
            keep = np.array([
                o >= 0 and len(by_strat[primary][int(o)]) < max_per_organ
                for o in oid_np
            ])
        if not keep.any():
            continue
        keep_t = torch.as_tensor(keep, dtype=torch.bool)
        pv  = batch['pixel_values'][keep_t].to(DEVICE)
        m   = batch['masks'][keep_t].to(DEVICE)
        oid = batch['organ_id'][keep_t].to(DEVICE).long()
        oid_np_kept = oid.cpu().numpy()
        B = pv.shape[0]
        for name, emb_fn in strategies.items():
            if emb_fn is None:
                out = model(pixel_values=pv, organ_id=oid, masks=m)
            else:
                emb = emb_fn(B, DEVICE)
                mod_list = fa.film_modulations_from_embedding(model, emb)
                with fa.inject_modulations(model, mod_list):
                    out = model(pixel_values=pv, organ_id=oid, masks=m)
            d = fa.compute_per_sample_dice(out['logits'], m).cpu().numpy()
            for organ_val, dice_val in zip(oid_np_kept, d):
                by_strat[name][int(organ_val)].append(float(dice_val))
        if bi % 25 == 0:
            counts = {int(o): len(by_strat[primary][int(o)]) for o in by_strat[primary]}
            print(f'  batch {bi:4d}: per-organ counts so far = {counts}')
    return by_strat

per_organ = eval_per_organ(strategies, max_per_organ=MAX_PER_ORGAN)

organs_present = sorted({o for v in per_organ.values() for o in v.keys()})
print(f'\nOrgans seen in test_ds: {[(o, class_to_organ_dict.get(o, str(o))) for o in organs_present]}')

mean_by_organ = {name: {o: float(np.mean(per_organ[name][o]))
                        for o in organs_present if per_organ[name].get(o)}
                 for name in strategies}

fig, ax = plt.subplots(figsize=(max(8, len(organs_present) * 1.4), 4.5))
n_strat = len(strategies)
width = 0.8 / n_strat
positions = np.arange(len(organs_present))
for si, name in enumerate(strategies):
    vals = [mean_by_organ[name].get(o, np.nan) for o in organs_present]
    ax.bar(positions + (si - (n_strat - 1) / 2) * width, vals, width, label=name)
ax.set_xticks(positions)
ax.set_xticklabels([class_to_organ_dict.get(o, str(o)) for o in organs_present], rotation=20)
ax.set_ylabel('mean Dice')
ax.set_ylim(0, 1)
ax.set_title(f'Per-organ Dice under OOD embedding strategies (FiLM, cap={MAX_PER_ORGAN}/organ)')
ax.legend(fontsize=8, loc='lower right', ncol=2)
ax.grid(axis='y', ls=':', alpha=.4)
fig.tight_layout(); fig.savefig(FIG_DIR/'7_ood_per_organ.png', dpi=130); plt.show()

tbl = pd.DataFrame(
    {name: [mean_by_organ[name].get(o, float('nan')) for o in organs_present]
     for name in strategies},
    index=[class_to_organ_dict.get(o, str(o)) for o in organs_present],
)
n_per_organ = {o: len(per_organ['baseline (true id)'][o]) for o in organs_present}
tbl.insert(0, 'n', [n_per_organ[o] for o in organs_present])
print(tbl.round(3))

results_ood = {name: float(np.mean(list(mean_by_organ[name].values())))
               for name in strategies}

## 8. Embedding interpolation

Linearly interpolate between two organ embeddings (in the first block's space) and reinterpret across all blocks.

In [ ]:
# For (organ_A, organ_B): traverse e = α·e_B + (1-α)·e_A, evaluate Dice on val images of both.
ALPHAS = np.linspace(0, 1, 9)
PAIRS = [(1, 3), (2, 5), (3, 4)]  # breast↔liver, cardiac↔kidney, thyroid↔fetal — adjust as desired
fig, axes = plt.subplots(1, len(PAIRS), figsize=(4*len(PAIRS), 3.5), sharey=True)
if len(PAIRS) == 1: axes = [axes]
for ax, (a, b) in zip(axes, PAIRS):
    e_a = torch.as_tensor(organ_W[a], device=DEVICE).unsqueeze(0)
    e_b = torch.as_tensor(organ_W[b], device=DEVICE).unsqueeze(0)
    dice_a, dice_b = [], []
    for alpha in ALPHAS:
        emb = (1-alpha) * e_a + alpha * e_b   # (1, D)
        def f_emb(B, dev, _e=emb): return _e.expand(B, -1).to(dev)
        # Evaluate separately on A-organ images and B-organ images
        @torch.inference_mode()
        def eval_for_organ(target_o):
            ds = []
            for bi, batch in enumerate(loader):
                if bi >= 10: break
                sel = batch['organ_id'] == target_o
                if sel.sum() == 0: continue
                sub = {k: v[sel] for k, v in batch.items() if torch.is_tensor(v)}
                pv = sub['pixel_values'].to(DEVICE); m = sub['masks'].to(DEVICE)
                B = pv.shape[0]
                e_sub = f_emb(B, DEVICE)
                mod_list = fa.attn_modulations_from_embedding(model, pv, e_sub, layer_configs)
                with fa.inject_modulations(model, mod_list):
                    out = model(pixel_values=pv, organ_id=sub['organ_id'].to(DEVICE).long(), masks=m)
                ds.append(fa.compute_per_sample_dice(out['logits'], m).cpu().numpy())
            return float(np.concatenate(ds).mean()) if ds else float('nan')
        dice_a.append(eval_for_organ(a))
        dice_b.append(eval_for_organ(b))
    ax.plot(ALPHAS, dice_a, 'o-', label=f'images: {class_to_organ_dict.get(a,a)}')
    ax.plot(ALPHAS, dice_b, 's-', label=f'images: {class_to_organ_dict.get(b,b)}')
    ax.set_xlabel(f'α  (0 = {class_to_organ_dict.get(a,a)}, 1 = {class_to_organ_dict.get(b,b)})')
    ax.set_title(f'{class_to_organ_dict.get(a,a)} ↔ {class_to_organ_dict.get(b,b)}', fontsize=10)
    ax.legend(fontsize=7)
axes[0].set_ylabel('Dice')
fig.suptitle('Linear embedding interpolation between two organs')
fig.tight_layout(); fig.savefig(FIG_DIR/'8_interpolation.png', dpi=130); plt.show()

In [ ]:
ALPHAS = np.linspace(0, 1, 9)
PAIRS = [(1, 3), (2, 5), (3, 4)]
fig, axes = plt.subplots(1, len(PAIRS), figsize=(4*len(PAIRS), 3.5), sharey=True)
if len(PAIRS) == 1: axes = [axes]
# A cleaner FiLM-aware interpolation: interpolate per-block, since each block's embedding is independent.
for ax, (a, b) in zip(axes, PAIRS):
    block_pairs = [(fl.embed.weight[a].detach(), fl.embed.weight[b].detach()) for fl in film_layers]
    dice_a_list, dice_b_list = [], []
    for alpha in ALPHAS:
        def f_emb_list(B, dev, _alpha=alpha):
            return [((1-_alpha)*e_a + _alpha*e_b).unsqueeze(0).expand(B, -1).to(dev) for (e_a, e_b) in block_pairs]
        @torch.inference_mode()
        def eval_for_organ(target_o):
            ds = []
            for bi, batch in enumerate(loader):
                if bi >= 10: break
                sel = batch['organ_id'] == target_o
                if sel.sum() == 0: continue
                sub = {k: v[sel] for k, v in batch.items() if torch.is_tensor(v)}
                pv = sub['pixel_values'].to(DEVICE); m = sub['masks'].to(DEVICE)
                B = pv.shape[0]
                emb_list = f_emb_list(B, DEVICE)
                # Build mod_list by running each block's MLP on its own (per-block) embedding
                mod_list = []
                for fl, e in zip(film_layers, emb_list):
                    bg = fl.mlp(e)
                    beta, gamma = bg.chunk(2, dim=-1)
                    mod_list.append((gamma.unsqueeze(-1).unsqueeze(-1), beta.unsqueeze(-1).unsqueeze(-1)))
                with fa.inject_modulations(model, mod_list):
                    out = model(pixel_values=pv, organ_id=sub['organ_id'].to(DEVICE).long(), masks=m)
                ds.append(fa.compute_per_sample_dice(out['logits'], m).cpu().numpy())
            return float(np.concatenate(ds).mean()) if ds else float('nan')
        dice_a_list.append(eval_for_organ(a))
        dice_b_list.append(eval_for_organ(b))
    ax.plot(ALPHAS, dice_a_list, 'o-', label=f'images: {class_to_organ_dict.get(a,a)}')
    ax.plot(ALPHAS, dice_b_list, 's-', label=f'images: {class_to_organ_dict.get(b,b)}')
    ax.set_xlabel(f'α  (0 = {class_to_organ_dict.get(a,a)}, 1 = {class_to_organ_dict.get(b,b)})')
    ax.set_title(f'{class_to_organ_dict.get(a,a)} ↔ {class_to_organ_dict.get(b,b)}', fontsize=10)
    ax.legend(fontsize=7)
axes[0].set_ylabel('Dice')
fig.suptitle('Per-block linear embedding interpolation (FiLM)')
fig.tight_layout(); fig.savefig(FIG_DIR/'8_interpolation.png', dpi=130); plt.show()

## 9. Task-analogy arithmetic

In [ ]:
# Same as Attn: use first FiLM block's organ embedding as the representative space.
from itertools import permutations
n_organs_trained = rep_W.shape[0] - 1
W = rep_W[:n_organs_trained]
results = []
for a, b, c in permutations(range(n_organs_trained), 3):
    v = W[a] - W[b] + W[c]
    sims = (W @ v) / (np.linalg.norm(W, axis=1) * np.linalg.norm(v) + 1e-12)
    nearest = int(sims.argmax())
    results.append((a, b, c, nearest, float(sims.max())))
import pandas as pd
df = pd.DataFrame(results, columns=['A','B','C','nearest','cos_sim'])
df['A_name'] = df['A'].map(class_to_organ_dict)
df['B_name'] = df['B'].map(class_to_organ_dict)
df['C_name'] = df['C'].map(class_to_organ_dict)
df['nearest_name'] = df['nearest'].map(class_to_organ_dict)
df['novel'] = df.apply(lambda r: r['nearest'] not in (r['A'], r['B'], r['C']), axis=1)
print('Fraction with novel nearest:', df['novel'].mean().round(3))
df.sort_values('cos_sim', ascending=False).head(15)

## 10. TCAV-on-FiLM

In [ ]:
TCAV_BATCHES = 10
EPS = 0.05
organs_for_tcav = [o for o in organs_used if (data['organ_id']==o).sum() >= 30]
tcav_scores = np.full((len(organs_for_tcav), len(layer_ids)), np.nan)
for oi, o in enumerate(organs_for_tcav):
    y = (data['organ_id'] == o).astype(int)
    for li_idx, lid in enumerate(layer_ids):
        X = data['layers'][lid]['gamma']
        # FiLM γ has zero within-organ variance, so CAV is determined by between-organ means.
        # We add tiny jitter so the SVM converges.
        Xj = X + 1e-6 * np.random.randn(*X.shape)
        try:
            cav_raw = fa.compute_cav(Xj, y)
        except Exception:
            continue
        positives = []
        seen = 0
        for bi, batch in enumerate(loader):
            if seen >= TCAV_BATCHES * BATCH: break
            sel = batch['organ_id'] == o
            if sel.sum() == 0: continue
            sub = {k: v[sel] for k, v in batch.items() if torch.is_tensor(v)}
            B = sub['pixel_values'].shape[0]
            seen += B
            mod_list_base = fa.build_mod_list_from_per_organ(
                torch.full((B,), o, dtype=torch.long),
                g_per_layer_per_organ, b_per_layer_per_organ,
                layer_configs, MAX_C, DEVICE,
            )
            cav_t = torch.as_tensor(cav_raw, dtype=torch.float32, device=DEVICE).view(1, -1, 1, 1)
            target_lid, n_ch = layer_configs[li_idx]
            pert = torch.nn.functional.adaptive_avg_pool1d(cav_t.view(1, -1), n_ch).view(1, n_ch, 1, 1)
            mod_plus = list(mod_list_base)
            mod_plus[li_idx] = (mod_list_base[li_idx][0] + EPS * pert, mod_list_base[li_idx][1])
            mod_minus = list(mod_list_base)
            mod_minus[li_idx] = (mod_list_base[li_idx][0] - EPS * pert, mod_list_base[li_idx][1])
            _, d_plus = fa.run_with_modulations(model, sub, mod_plus, DEVICE)
            _, d_minus = fa.run_with_modulations(model, sub, mod_minus, DEVICE)
            dd = (d_plus - d_minus).cpu().numpy() / (2 * EPS)
            positives.append((dd > 0).astype(int))
        if positives:
            tcav_scores[oi, li_idx] = float(np.concatenate(positives).mean())
    print(f'organ {o} done')
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(tcav_scores, vmin=0, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(layer_ids))); ax.set_xticklabels(layer_ids)
ax.set_yticks(range(len(organs_for_tcav))); ax.set_yticklabels([class_to_organ_dict.get(o, str(o)) for o in organs_for_tcav])
ax.set_xlabel('layer'); ax.set_ylabel('organ concept')
ax.set_title('TCAV (FiLM): fraction of samples with +∂Dice/∂γ along CAV (0.5 = no signal)')
fig.colorbar(im, ax=ax); fig.tight_layout(); fig.savefig(FIG_DIR/'10_tcav.png', dpi=130); plt.show()

## 11. Summary

In [ ]:
_organ_ids_valid = [i for i in np.unique(data['organ_id']) if i >= 0]
_mean_dice = np.mean([data['dice'][data['organ_id'] == i].mean() for i in _organ_ids_valid])
print('=== Summary (FiLM UNet) ===')
print(f'Samples analysed       : {len(data["organ_id"])}')
print(f'Mean baseline Dice     : {_mean_dice:.4f}  (mean of organ means)')
print('Per-layer probe accuracy (5-fold):')
for lid in layer_ids:
    print(f'  L{lid:2d}: {probe_acc[lid]["acc"]:.3f}')
print('Channels with cross-organ Var > 1e-3:')
for lid in layer_ids:
    print(f'  L{lid:2d}: {(var_per_layer[lid] > 1e-3).mean():.2%}')
print('OOD strategies mean Dice:')
for k, v in results_ood.items():
    print(f'  {k}: {v:.4f}')
print(f'\nFigures saved to {FIG_DIR.resolve()}')

## 12. Learn a custom organ embedding

Same setup as the Attn notebook: freeze the whole UNet and learn a single `(1, film_embed)` vector that, when reinterpreted by every FiLM2d block's MLP, minimises `DiceBCELoss` on the augmented `val_cls` split.

A note on capacity: this gives the optimiser only **`film_embed` parameters** (64 here) to fit the entire dataset — much tighter than the Attn version's 768-dim space. If the loss barely moves that's itself informative: it tells you a single 64-dim vector can't simultaneously satisfy all organs through these frozen MLPs. A per-block alternative is mentioned at the bottom.

In [ ]:
from nets.segm_net import DiceBCELoss

# 1. Freeze the entire model; keep eval mode (deterministic dropout / norm).
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

# 2. Single trainable (1, film_embed) vector — initialised at the empirical mean
#    of the first block's organ rows so we start inside the model's manifold.
custom_emb = torch.nn.Parameter(
    torch.as_tensor(mu_emp, dtype=torch.float32, device=DEVICE).unsqueeze(0)
)
optimizer = torch.optim.Adam([custom_emb], lr=1e-3)
criterion = DiceBCELoss()

# 3. Training dataset with augmentations
train_ds_custom = USdatasetOmni(
    DATA_DIR, split='train_cls',
    transforms=get_sft_transforms(train=True, size=IMG_SIZE),
    data_type='segmentation',
    out_size=IMG_SIZE,
    keep_aspect_ratio=True,
    self_norm=True,
)
train_loader_custom = DataLoader(
    train_ds_custom, batch_size=BATCH, shuffle=True, num_workers=4,
    pin_memory=DEVICE == 'cuda',
)
print(f'Training samples: {len(train_ds_custom)}')

# 4. Gradient-enabled re-implementation of film_modulations_from_embedding.
#    The helper version has @torch.no_grad which would block grad flow.
def compute_mods_with_grad(emb):
    """emb: (B, film_embed). Each block's MLP reinterprets the same vector.
    Returns mod_list of (γ, β) shaped (B, n_channels, 1, 1) — grad-tracking."""
    mod_list = []
    for fl in film_layers:
        beta_gamma = fl.mlp(emb)
        beta, gamma = beta_gamma.chunk(2, dim=-1)
        beta  = beta.unsqueeze(-1).unsqueeze(-1)
        gamma = gamma.unsqueeze(-1).unsqueeze(-1)
        mod_list.append((gamma, beta))
    return mod_list

# 5. Training loop
N_EPOCHS = 3
LOG_EVERY = 20
losses = []
for ep in range(N_EPOCHS):
    epoch_losses = []
    for bi, batch in enumerate(train_loader_custom):
        pv  = batch['pixel_values'].to(DEVICE)
        m   = batch['masks'].to(DEVICE)
        oid = batch['organ_id'].to(DEVICE).long()
        B   = pv.shape[0]

        emb_b    = custom_emb.expand(B, -1)
        mod_list = compute_mods_with_grad(emb_b)
        with fa.inject_modulations(model, mod_list):
            out = model(pixel_values=pv, organ_id=oid, masks=m)
        loss = criterion(out['logits'], m)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        epoch_losses.append(loss.item())
        if bi % LOG_EVERY == 0:
            print(f'  epoch {ep} batch {bi:4d}  loss={loss.item():.4f}')
    print(f'epoch {ep} mean loss: {np.mean(epoch_losses):.4f}')

# 6. Loss curve
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, lw=1)
ax.set_xlabel('iteration'); ax.set_ylabel('DiceBCE loss')
ax.set_title('Training a custom organ embedding (FiLM UNet, model frozen)')
ax.grid(ls=':', alpha=.5)
fig.tight_layout(); fig.savefig(FIG_DIR/'custom_emb_loss.png', dpi=130); plt.show()

# 7. Where did the learned embedding land relative to the trained organs
#    (in the first block's embedding space)?
final_emb = custom_emb.detach().cpu().numpy().squeeze()
print(f'\nFinal custom embedding shape: {final_emb.shape}')
print("L2 distance from each trained organ (first FiLM block's space):")
for i in range(rep_W.shape[0]):
    d = float(np.linalg.norm(final_emb - rep_W[i]))
    name = class_to_organ_dict.get(i, 'unknown' if i == rep_W.shape[0]-1 else f'id{i}')
    print(f'  {name:15s}: {d:.4f}')

# 8. Per-organ evaluation of the learned embedding, side-by-side with baselines.
def emb_custom(B_, dev):
    return custom_emb.detach().expand(B_, -1).to(dev)

strategies_eval = {**strategies, 'custom (learned)': emb_custom}
per_organ_eval  = eval_per_organ(strategies_eval, max_per_organ=MAX_PER_ORGAN)
organs_eval     = sorted({o for v in per_organ_eval.values() for o in v.keys()})
mean_eval = {name: {o: float(np.mean(per_organ_eval[name][o]))
                    for o in organs_eval if per_organ_eval[name].get(o)}
             for name in strategies_eval}

fig, ax = plt.subplots(figsize=(max(8, len(organs_eval) * 1.5), 4.5))
n_strat = len(strategies_eval)
width = 0.8 / n_strat
positions = np.arange(len(organs_eval))
for si, name in enumerate(strategies_eval):
    vals = [mean_eval[name].get(o, np.nan) for o in organs_eval]
    ax.bar(positions + (si - (n_strat - 1) / 2) * width, vals, width, label=name)
ax.set_xticks(positions)
ax.set_xticklabels([class_to_organ_dict.get(o, str(o)) for o in organs_eval], rotation=20)
ax.set_ylabel('mean Dice'); ax.set_ylim(0, 1)
ax.set_title('Per-organ Dice — baselines + learned custom embedding (FiLM)')
ax.legend(fontsize=8, loc='lower right', ncol=2)
ax.grid(axis='y', ls=':', alpha=.4)
fig.tight_layout(); fig.savefig(FIG_DIR/'custom_emb_per_organ.png', dpi=130); plt.show()

In [ ]:
strategies_eval = {**strategies, 'custom (learned)': emb_custom}
per_organ_eval  = eval_per_organ(strategies_eval, max_per_organ=MAX_PER_ORGAN)
organs_eval     = sorted({o for v in per_organ_eval.values() for o in v.keys()})
mean_eval = {name: {o: float(np.mean(per_organ_eval[name][o]))
                    for o in organs_eval if per_organ_eval[name].get(o)}
             for name in strategies_eval}

fig, ax = plt.subplots(figsize=(max(8, len(organs_eval) * 1.5), 4.5))
n_strat = len(strategies_eval)
width = 0.8 / n_strat
positions = np.arange(len(organs_eval))
for si, name in enumerate(strategies_eval):
    vals = [mean_eval[name].get(o, np.nan) for o in organs_eval]
    ax.bar(positions + (si - (n_strat - 1) / 2) * width, vals, width, label=name)
ax.set_xticks(positions)
ax.set_xticklabels([class_to_organ_dict.get(o, str(o)) for o in organs_eval], rotation=20)
ax.set_ylabel('mean Dice'); ax.set_ylim(0, 1)
ax.set_title('Per-organ Dice — baselines + learned custom embedding (FiLM)')
ax.legend(fontsize=8, loc='lower right', ncol=2)
ax.grid(axis='y', ls=':', alpha=.4)
fig.tight_layout(); fig.savefig(FIG_DIR/'custom_emb_per_organ.png', dpi=130); plt.show()